In [2]:
import numpy as np
import pandas as pd
import os

In [3]:
class Piece:

    rotation = 1
    flipped = False
    placed = False
    pattern = None # This will be a string to help define the patterns of connections

    def __init__(self, fileLocation=None, name=None):
        self.fileLocation = fileLocation
        if fileLocation is None:
            self.name = name
        else:
            self.name = os.path.splitext(os.path.basename(fileLocation))[0]    
        self.createPiece()

    def createPiece(self):
        if self.fileLocation is None:
            self.data = pd.DataFrame({'orientation':True, 'unit':None, 'edge_1':None, 'edge_2':None, 'edge_3':None}, index=[1])
        else:
            df = pd.read_csv("Resources/Board.csv")
            df = df.set_index("id")
            df["unit"] = self.name
            self.data = df

    def flipPiece(self):
        pass

    def rotatePiece(self):
        pass

    def triangleCount(self):
        return len(self.data)

In [13]:
"""
- All Triangle connections are in a "flipped" order. Two triangles with the same orientation can't connect on any edges. 
- This is orientation will be included in the triangle creation.
- This also means we can create a "true" orientation for triangles when placed/joined: 
    Default Orientation: Point Facing Upwards
- On this Default Orientation define the edges in the order: Left(0), Right(1), Bottom(2)
- And if you look at some examples of this orientation you will see that it is the same orientations that join together.
    For Example: The Bottoms(2) touch, the Lefts(0) touch, and the Rights(1) touch.
"""

class Triangle:

    # This flag is used for checking/searching through Pieces.
    checked = False

    def __init__(self, id, orientation, edge_0=None, edge_1=None, edge_2=None, unit=None):
        self.id = id
        self.orientation = orientation
        self.edges = [edge_0, edge_1, edge_2]
        self.unit = unit
        self.connectEdges()

    def printEdge(self, n):
        if self.edges[n] is None:
            return "None"
        else:
            return f"{self.edges[n].id}"

    def __str__(self):
        return f"Triangle {self.id} | [{self.printEdge(0)}, {self.printEdge(1)}, {self.printEdge(2)} | {self.orientation}]"
    
    def __repr__(self):
        return f"Triangle {self.id} | [{self.printEdge(0)}, {self.printEdge(1)}, {self.printEdge(2)}] | Unit: {self.unit}"
    
    def setEdge(self, n, edge):
        self.edges[n] = edge

    def connectEdges(self):
        for n, edge in enumerate(self.edges):
            if edge:
                if self.orientation == edge.orientation: raise ValueError(f"There is an error with connecting {self.id} {self.unit} with {edge.id} {edge.unit}")
                edge.setEdge(n, self)
            

class Piece:

    rotation = 1
    flipped = False
    solvedTemp = False
    placed = False

    def __init__(self, fileLocation=None, name=None):
        self.fileLocation = fileLocation
        if fileLocation is None:
            self.triangleSet = {1: Triangle(1, True, unit=name)}
        else:
            self.name = os.path.splitext(os.path.basename(fileLocation))[0]
            self.createPiece()

    def __repr__(self):
        return f"Piece {self.name} | {self.placed}"
        
    def createPiece(self):
        self.triangleSet = {None:None}
        triangleInfo = pd.read_csv(self.fileLocation, dtype={"id":"Int64", "orientation":bool, "edge_0":"Int64", "edge_1":"Int64", "edge_2":"Int64"})
        triangleInfo = triangleInfo.replace({np.nan: None}) #triangleInfo.where(pd.notnull(triangleInfo), None)
        for n in range(len(triangleInfo)):
            triangle = triangleInfo.iloc[n].copy()
            if triangle.edge_0 and triangle.edge_0 >= triangle.id: triangle.edge_0 = None
            if triangle.edge_1 and triangle.edge_1 >= triangle.id: triangle.edge_1 = None
            if triangle.edge_2 and triangle.edge_2 >= triangle.id: triangle.edge_2 = None
            #print(triangle.edge_0, triangle.edge_1, triangle.edge_2)
            self.triangleSet[triangle.id] = Triangle(id = triangle.id,
                                               orientation = triangle.orientation,
                                               edge_0 = self.triangleSet[triangle.edge_0],
                                               edge_1 = self.triangleSet[triangle.edge_1],
                                               edge_2 = self.triangleSet[triangle.edge_2],
                                               unit = self.name
                                              )
        self.triangleSet.pop(None)

    def rotatePiece(self):
        # Let's always rotate once clockwise (There are 6 total rotations before back to start).
        self.rotation += 1
        for triangle in self.triangleSet.values():
            triangle.orientation = not triangle.orientation
            # Each orientation changes the edge assignments by subtracting one.
            triangle.edges[0], triangle.edges[1], triangle.edges[2] = triangle.edges[1], triangle.edges[2], triangle.edges[0] 

    def flipPiece(self):
        self.flipped = not self.flipped
        for triangle in self.triangleSet.values():
            triangle.edges[0], triangle.edges[1], triangle.edges[2] = triangle.edges[1], triangle.edges[0], triangle.edges[2] 


    def triangleCount(self):
        return  len(self.triangleSet)
    
    def resetTriangleSearch(self):
        for triangle in self.triangleSet.values():
            triangle.checked = False

    def printPieces(self):
        for indx in range(1, self.triangleCount() + 1):
            print(f" Triangle {indx} : {self.triangleSet[indx]}")
            # print(f" Triangle {indx} : {self.triangleSet[indx].unit}")

    def isSolved(self):
        for indx in range(1, self.triangleCount() + 1):
            if self.triangleSet[indx].unit == "Board":
                return False
        return True
    


In [14]:
def setupGame():
    # Create Board
    board = Piece("Resources/Board.csv")
    # Loop Through and Create Pieces
    folderPath = "Resources/GamePieces/"
    pieces = {}
    for fileName in os.listdir(folderPath):
        filePath = os.path.join(folderPath, fileName)
        if os.path.isfile(filePath):
            pieces[fileName] = (Piece(filePath))
    # Create the Seven Triangles (Obstacles) that are set by the Dice.
    obstacles = []
    for setPiece in range(1,8):
        obstacles.append(Piece(name="Obstacle" + str(setPiece)))
    return board, pieces, obstacles


def checkValidPlacement(board, boardTriangleId, piece, pieceTriangleId=1, debug=False):
    boardTriangle = board.triangleSet[boardTriangleId]
    pieceTriangle = piece.triangleSet[pieceTriangleId]
    if debug: print(pieceTriangle, boardTriangle)
    if boardTriangle is None: # Is it Off the Board.
        if debug: print("Off the Board")
        piece.resetTriangleSearch()
        return False 
    if boardTriangle.orientation != pieceTriangle.orientation: # If Different Orientations.
        if debug: print("Different Orientation")
        piece.resetTriangleSearch()
        return False
    if boardTriangle.unit != "Board": # No a Piece is there.
        if debug: print(f"Already Occupied by {boardTriangle.unit}")
        piece.resetTriangleSearch()
        return False
    pieceTriangle.checked = True
    for num, edge in enumerate(pieceTriangle.edges):
        if edge is None:
            pass
        elif not edge.checked:
            nextBoardTriangle = board.triangleSet[boardTriangleId].edges[num]
            if nextBoardTriangle is None:
                piece.resetTriangleSearch()
                return False
            if not checkValidPlacement(board, nextBoardTriangle.id, piece, pieceTriangle.edges[num].id, debug):
                return False
    return True


def placePiece(board, boardTriangleId, piece, pieceTriangleId=1):
    boardTriangle = board.triangleSet[boardTriangleId]
    pieceTriangle = piece.triangleSet[pieceTriangleId]
    # Place the Piece.
    boardTriangle.unit = pieceTriangle.unit
    pieceTriangle.checked = True
    for num, edge in enumerate(pieceTriangle.edges):
        if edge is None:
            pass
        elif not edge.checked:
            nextBoardTriangle = board.triangleSet[boardTriangleId].edges[num]
            if nextBoardTriangle is None:
                piece.resetTriangleSearch()
                continue 
            else:
                placePiece(board, nextBoardTriangle.id, piece, edge.id)


def moveThroughPiece(board, boardTriangleId, piece, pieceTriangleId=1, debug=False):
    boardTriangle = board.triangleSet[boardTriangleId]
    pieceTriangle = piece.triangleSet[pieceTriangleId]
    if debug: print(pieceTriangle, boardTriangle)
    if boardTriangle is None: # Is it Off the Board. (Maybe Removable due to the same condition below)
        if debug: print("Off the Board")
        piece.resetTriangleSearch()
        return False 
    if boardTriangle.orientation != pieceTriangle.orientation: # If Different Orientations.
        if debug: print("Different Orientation")
        piece.resetTriangleSearch()
        return False
    if boardTriangle.unit != "Board": # No a Piece is there.
        if debug: print(f"Already Occupied by {boardTriangle.unit}")
        piece.resetTriangleSearch()
        return False
    pieceTriangle.checked = True
    for num, edge in enumerate(pieceTriangle.edges):
        if edge is None:
            pass
        elif not edge.checked:
            nextBoardTriangle = board.triangleSet[boardTriangleId].edges[num]
            if nextBoardTriangle is None:
                piece.resetTriangleSearch()
                return False
            if not moveThroughPiece(board, nextBoardTriangle.id, piece, edge.id, debug):
                return False
    return True


def removePiece(board, piece):
    for id in range(1, board.triangleCount() + 1):
        boardTriangle = board.triangleSet[id]
        # Remove the piece if it's the correct unit.
        if boardTriangle.unit == piece.name:
            boardTriangle.unit = "Board"
        boardTriangle.checked = True
    piece.resetTriangleSearch()
    return board

        
def rollDice(board, obstacles):
    rand1 = np.random.choice([12, 13, 23, 24, 32, 33, 41, 42])
    placePiece(board, rand1, obstacles[0])
    rand2 = np.random.choice([2, 4, 7, 8, 9, 11, 16, 17])
    placePiece(board, rand2, obstacles[1])
    rand3 = np.random.choice([25, 26, 36, 37, 38, 40, 45, 47])
    placePiece(board, rand3, obstacles[2])
    rand4 = np.random.choice([10,27,31])
    placePiece(board, rand4, obstacles[3])
    rand5 = np.random.choice([18, 22, 39])
    placePiece(board, rand5, obstacles[4])
    rand6 = np.random.choice([1, 5, 15, 34, 44, 48])
    placePiece(board, rand6, obstacles[5])
    rand7 = np.random.choice([19, 20, 21, 28, 29, 30])
    placePiece(board, rand7, obstacles[6])
    print(f"The Dice Rolls are {rand1}, {rand2}, {rand3}, {rand4}, {rand5}, {rand6}, {rand7}")


In [15]:
def reorderPices(pieces):
    return [pieces["DarkGreen5.csv"], pieces["Lavender5.csv"], pieces["LightGreen5.csv"], pieces["Red5.csv"], 
            pieces["Purple4.csv"], pieces["Orange4.csv"], pieces["Pink4.csv"], pieces["Blue3_1.csv"], pieces["Blue3_2.csv"], pieces["Yellow2.csv"], pieces["DarkBlue1.csv"]]

def setPieceConfigs(pieces):
    # DarkGreen5
    pieces["DarkGreen5.csv"].maxRotation = 6
    pieces["DarkGreen5.csv"].canFlip = False
    # Lavender5
    pieces["Lavender5.csv"].maxRotation = 6
    pieces["Lavender5.csv"].canFlip =  True
    # LightGreen5
    pieces["LightGreen5.csv"].maxRotation = 6
    pieces["LightGreen5.csv"].canFlip = True
    # Red5
    pieces["Red5.csv"].maxRotation = 6
    pieces["Red5.csv"].canFlip = False
    # Purple4
    pieces["Purple4.csv"].maxRotation = 2
    pieces["Purple4.csv"].canFlip =  False
    # Orange4
    pieces["Orange4.csv"].maxRotation = 6
    pieces["Orange4.csv"].canFlip = False
    # Pink4
    pieces["Pink4.csv"].maxRotation = 6
    pieces["Pink4.csv"].canFlip =  False
    # Blue3_1
    pieces["Blue3_1.csv"].maxRotation = 3
    pieces["Blue3_1.csv"].canFlip = True
    # Blue3_2
    pieces["Blue3_2.csv"].maxRotation = 3
    pieces["Blue3_2.csv"].canFlip = True
    # Yellow2
    pieces["Yellow2.csv"].maxRotation = 4
    pieces["Yellow2.csv"].canFlip =  False
    # DarkBlue1
    pieces["DarkBlue1.csv"].maxRotation = 2
    pieces["DarkBlue1.csv"].canFlip = False


In [16]:
def solvePuzzle(board, pieces):
    """ 
    Recursive Function that uses the board and available pieces as it's input
    In each call it takes a piece and brute force tries it at every triangle. Then rotates it. 
    If it ever succeeds it calls itself with 1 less piece. If it fails it returns itself witht the piece
    still available, then it continues the previous pieces brute force.
    """
    for piece in pieces:
        if not piece.placed:
            break
    if piece.placed:
        if board.isSolved():
            board.solvedTemp = True
            return board, pieces
        return board, pieces
    for indx in range(1, board.triangleCount() + 1):
        if checkValidPlacement(board, indx, piece):
            if piece.name == "LightGreen5": print("Placed ", indx)
            # print(piece, indx)
            piece.resetTriangleSearch() # Current limitation is that this doesn't happen for valid returns
            placePiece(board, indx, piece)
            piece.placed = True
            piece.resetTriangleSearch()
            board, pieces = solvePuzzle(board, pieces)
            if board.solvedTemp:
                return board, pieces
            removePiece(board, piece)
            piece.placed = False
    piece.rotatePiece()
    if piece.name == "LightGreen5": print("Rotated ", piece.rotation)
    if piece.rotation <= 6:
        board, pieces = solvePuzzle(board, pieces)
        if board.solvedTemp:
            return board, pieces
        # removePiece(board, piece)
        # piece.placed = False
    piece.rotation = 1
    if not piece.flipped:
        if piece.name == "LightGreen5": print("Flipped ", piece.flipped)
        piece.flipPiece()
        board, pieces = solvePuzzle(board, pieces)
        if board.solvedTemp:
            return board, pieces
        piece.flipPiece()
    return board, pieces

In [20]:
board, pieces, obstacles = setupGame() # Build the Board and Pieces.
piece = pieces["Blue3_1.csv"]
piece.rotatePiece()
piece.rotatePiece()

In [25]:
from timeit import time

n = 1000000
total = 0
for i in range(n):
    start = time.time()
    placement = checkValidPlacement(board, 1, piece)
    end = time.time()
    total += end-start
print(placement)
total

True


0.9715025424957275

In [ ]:
board, pieces, obstacles = setupGame() # Build the Board and Pieces.
pieces = reorderPices(pieces)
rollDice(board, obstacles) # Roll the Dice and Add the Obstacles to the Board.
solvedBoard, pieces = solvePuzzle(board, pieces)
solvedBoard.printPieces()

The Dice Rolls are 24, 2, 45, 31, 39, 34, 30
Placed  6
Placed  17
Placed  25
Rotated  2
Placed  7
Placed  18
Rotated  3
Placed  19
Placed  29
Placed  42
Rotated  4
Placed  18
Placed  28
Rotated  5
Placed  25
Placed  27
Placed  36
Rotated  6
Placed  26
Placed  35
Rotated  7
Flipped  False
Placed  19
Placed  33
Rotated  2
Placed  18
Placed  20
Placed  28
Rotated  3
Placed  27
Placed  36
Placed  38
Rotated  4
Placed  26
Placed  35
Placed  47
Rotated  5
Placed  6
Placed  17
Placed  25
Rotated  6
Placed  7
Placed  16
Placed  18
Rotated  7
Flipped  False
Placed  19
Placed  33
Rotated  2
Placed  18
Placed  20
Placed  28
Rotated  3
Placed  27
Placed  36
Placed  38
Rotated  4
Placed  26
Placed  35
Placed  47
Rotated  5
Placed  6
Placed  17
Placed  25
Rotated  6
Placed  7
Placed  16
Placed  18
Rotated  7
Flipped  False
Placed  19
Placed  33
Rotated  2
Placed  18
Placed  20
Placed  28
Rotated  3
Placed  27
Placed  36
Placed  38
Rotated  4
Placed  26
Placed  35
Placed  47
Rotated  5
Placed  6
Plac

In [146]:
board, pieces, obstacles = setupGame() # Build the Board and Pieces.
pieces = reorderPices(pieces)

In [95]:
startingPoint = 21
piece = pieces[3]
if moveThroughPiece(board, startingPoint, piece):
    piece.resetTriangleSearch()
    print("Valid")
    placePiece(board, startingPoint, piece)

Valid


In [177]:
solvedBoard.solvedTemp, pieces

(False,
 [Piece DarkGreen5 | False,
  Piece Lavender5 | False,
  Piece LightGreen5 | False,
  Piece Red5 | False,
  Piece Purple4 | False,
  Piece Orange4 | False,
  Piece Pink4 | False,
  Piece Blue3_1 | False,
  Piece Blue3_2 | False,
  Piece Yellow2 | False,
  Piece DarkBlue1 | False])

In [111]:
solvedBoard.solvedTemp
listy = [1,2,3,4]
listy.insert(0, 4)
listy

[4, 1, 2, 3, 4]

In [14]:
df = pd.read_csv("Resources/Board.csv")
df = df.set_index("id")
df["unit"] = "Board"
df

,orientation,edge_0,edge_1,edge_2,unit
id,,,,,
1,True,NaN,NaN,3.0,Board
2,True,NaN,3.0,9.0,Board
3,False,4.0,2.0,1.0,Board
4,True,3.0,NaN,11.0,Board
5,False,6.0,NaN,NaN,Board
6,True,5.0,7.0,16.0,Board
7,False,8.0,6.0,NaN,Board
8,True,7.0,9.0,18.0,Board
9,False,10.0,8.0,2.0,Board


In [84]:
#Unit Tests
board, pieces, obstacles = setupGame()

if checkValidPlacement(board, 12, pieces[8]) != False: print("Failed Test 1")
if checkValidPlacement(board, 8, pieces[8]) != True: print("Failed Test 2")
pieces[8].resetTriangleSearch()
if checkValidPlacement(board, 2, pieces[8]) != False: print("Failed Test 3")
if checkValidPlacement(board, 34, pieces[8]) != True: print("Failed Test 4")
pieces[8].resetTriangleSearch()

Board  Triangle 13 | [14, 12, None]
Board  Triangle 14 | [13, 15, 24]
Board  Triangle 15 | [None, 14, None]
Board  None
Board  Triangle 9 | [10, 8, 2]
Board  Triangle 10 | [9, 11, 20]
Board  Triangle 11 | [12, 10, 4]
Board  Triangle 12 | [11, 13, 22]
Board  Triangle 3 | [4, 2, 1]
Board  Triangle 4 | [3, None, 11]
Board  None
Board  Triangle 35 | [36, 34, 25]
Board  Triangle 36 | [35, 37, None]
Board  Triangle 37 | [38, 36, 27]
Board  Triangle 38 | [37, 39, 45]


In [83]:
checkValidPlacement(board, 2, pieces[8], debug=True)

Triangle 1 | [None, 2, None] Triangle 2 | [None, 3, 9]
Board  Triangle 3 | [4, 2, 1]
Triangle 2 | [3, 1, None] Triangle 3 | [4, 2, 1]
Board  Triangle 4 | [3, None, 11]
Triangle 3 | [2, 4, None] Triangle 4 | [3, None, 11]
Board  None


False